# 01 — EDA de justificación del preprocesamiento ECoG

Este notebook describe el **desarrollo** de S1/S2/S3 y justifica con gráficas cada decisión del preprocesamiento: exclusión fija de canales, referencia de mediana común, banda 40–200 Hz, notch 60/120/180 Hz, Morlet (40 frecuencias, 7 ciclos), decimación a 100 Hz con antialias, recuperación del guante a 25 Hz e interpolación cúbica, retardo de 200 ms, escalado por fold y validación por bloques.

**Reglas**
- Las transformaciones se aplican **en memoria** con `tt_ecog.preprocessing` (la misma implementación que usará el notebook 02) y solo para graficar; no se guardan datos procesados.
- `train_dg` de desarrollo se usa **solo de forma descriptiva**: las gráficas son confirmatorias y no ajustan parámetros, que siguen fijos por protocolo. Las etiquetas de test **nunca** se leen.
- La exclusión de canales vive solo en `configs/experiment_config.yaml` (`channels.excluded`).

**Flujo VS Code + Colab**
1. Edita y haz commit/push en tu máquina local (las salidas del notebook las quita nbstripout al commitear).
2. Selecciona el kernel **Colab** en VS Code y ejecuta las celdas: la primera hace `git pull` en el runtime y muestra el commit, que debe coincidir con tu `git rev-parse HEAD` local.
3. Cada figura se guarda como PNG en `MyDrive/TT_ECoG/results/eda/<run_id>/fig/` junto con tablas y manifiestos, y se muestra aquí mismo; al guardar el notebook local las gráficas quedan en él.

Si el repositorio es privado, autentica Git de forma interactiva; no escribas tokens en celdas ni URLs.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

## Entorno
Se instalan solo las dependencias del EDA que falten (`requirements-eda.txt`: incluye MNE y scikit-learn). No se instala PyTorch ni CUDA. Si un módulo cambia después de importarlo, reinicia el runtime.

In [ ]:
subprocess.run([sys.executable, str(PROJECT / 'scripts/check_eda_environment.py'), '--install-missing'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '->', tt_ecog.__file__)

## Drive y nueva corrida
Puedes usar `Colab: Mount Google Drive to Server...` o ejecutar la celda. Las rutas salen del YAML: datos en `TT_ECoG/raw`, resultados en `TT_ECoG/results/eda/<run_id>/`. Cada ejecución crea una carpeta nueva; nunca se sobrescribe evidencia previa.

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
from tt_ecog.eda import finish_run, load_subjects, sections, show, start_run
CONFIG = PROJECT / 'configs/experiment_config.yaml'
ctx = start_run(CONFIG, DRIVE_ROOT)
print('Corrida:', ctx.run_id)
print('Resultados en Drive:', ctx.output)
print('Commit registrado:', ctx.env['git_commit'], '| cambios sin commitear:', ctx.env['git_dirty'])

## s01 · Datos de desarrollo
Carga `train_data` y `train_dg` de S1/S2/S3 (solo lectura, con hash antes y después) y muestra integridad, mapa crudo y trazas. Datos ausentes → `BLOCKED`; contrato inválido → `FAIL`.

In [ ]:
subjects = load_subjects(ctx)
show(sections.s01_overview.run(ctx, subjects))

## s02 · Canales atípicos → exclusión fija
Variabilidad local por canal y en el tiempo, ranking pico/típico, episodios y PSD. Las figuras `s02_S1_c054`, `s02_S2_c020` y `s02_S2_c037` son la evidencia que cita el YAML para excluir esos canales **antes** de la CMR. Las líneas 10×/20×/50× son referencias descriptivas, no umbrales.

In [ ]:
show(sections.s02_channels.run(ctx, subjects))

## s03 · Referencia de mediana común (CMR)
Compara media (CAR) y mediana (CMR), con todos los canales y solo con los conservados, durante los peores episodios; robustez en todo el registro; espectro, fracción de línea y correlación entre canales antes/después de la CMR.

In [ ]:
show(sections.s03_reference.run(ctx, subjects))

## s04 · Banda 40–200 Hz
Espectro 1–500 Hz, cambio movimiento/reposo y ERSP alrededor de inicios de movimiento (etiquetas de desarrollo, descriptivo), y diseño del pasabanda.

In [ ]:
show(sections.s04_band.run(ctx, subjects))

## s05 · Notch 60/120/180 Hz
Espectros de alta resolución alrededor de cada línea, estacionariedad de la línea de 60 Hz, fracción de línea por etapa (S2 muy contaminado) y respuesta del notch.

In [ ]:
show(sections.s05_notch.run(ctx, subjects))

## s06 · Morlet: 40 frecuencias, 7 ciclos, zero_mean
Familia de wavelets y resolución tiempo-frecuencia, escalograma con flexión, efecto de zero_mean y de los bordes (base de las máscaras).

In [ ]:
show(sections.s06_morlet.run(ctx, subjects))

## s07 · Decimación a 100 Hz con antialias
Ancho de banda de la envolvente de potencia, submuestreo directo frente a antialias, coherencia alta gamma–flexión y respuesta del filtro.

In [ ]:
show(sections.s07_decimation.run(ctx, subjects))

## s08 · Guante: 25 Hz y spline cúbico
Identificación del patrón de almacenamiento a 1 kHz (método y fase), recuperación de las muestras efectivas, espectro del guante y validación de la interpolación con muestras ocultas.

In [ ]:
show(sections.s08_glove.run(ctx, subjects))

## s09 · Retardo de 200 ms
Correlación alta gamma–flexión en función del retardo: los picos en retardos positivos (ECoG adelanta) sostienen ECoG(t) → flexión(t+0.2 s).

In [ ]:
show(sections.s09_delay.run(ctx, subjects))

## s10 · Escalado por fold
Colas pesadas de la potencia, STD frente a la escala de RobustScaler(10, 90), deriva entre bloques y MinMax ajustado sin el bloque evaluado.

In [ ]:
show(sections.s10_scaling.run(ctx, subjects))

## s11 · Validación por bloques y soporte temporal
Bloques B1–B5 (validación de la etapa 1 sombreada), autocorrelación de flexión y potencia, y soporte medido de la cadena frente al candidato de purga. La purga sigue pendiente hasta el paso 3.

In [ ]:
show(sections.s11_validation.run(ctx, subjects))

## Cierre
Escribe `run_summary.json`, `index.md` y `artifact_manifest.json` (SHA-256 de cada archivo) en la carpeta de la corrida. Falla si alguna sección no terminó en `COMPLETE`. Guarda el notebook local para conservar las gráficas y anota el `run_id` en `docs/experimental_log.md`.

In [ ]:
summary = finish_run(ctx)
print('Estado:', summary['status'])
print('Carpeta:', ctx.output)